# Anvendt KI og ML: Innsendingsoppgave – Salgsprediksjoner
Velkommen til denne Jupyter notebooken, som du skal bruke til den første obligatoriske innsendingsoppgaven i Høyskolen Kristianias emne «Anvendt kunstig intelligens og maskinlæring».

## Hvordan du bruker denne notebooken 
Du skal gå gjennom denne notebooken for å løse innsendingsoppgaven. Underveis skal du skrive kode som løser hver av de 7 kodeoppgavene skrevet _i kursiv_. Skriv kode inn i kodecellen rett under oppgaven, og legg gjerne til flere kodeceller. Du kan finne og gjenbruke relevant kode i notebooks i emnets modul 1, 2 og 3, og her i oppgaveteksten. Du kan eventuelt bruke andre programmeringsspråk etter avtale med veilederen din, men alle kodehint er gitt for Python.

Underveis vil du også finne 13 fritekstoppgaver du må besvare tekstlig. Også disse er skrevet _i kursiv_. De skal besvares i markdowncellene rett under oppgavene. Referer til egen kode der det er relevant.

Hvis du står fast, kontakt veilederen din for å få hjelp til å komme videre (via «Innboks» i hovedmenyen i Canvas, _ikke_ klikk «Lever oppgave»).

Etter at du har lagt inn egen tekst og kode for hver oppgave, skal du levere inn en PDF-versjon av denne notebooken. PDF-en skal vise både den skriftlige besvarelsen din og output fra alle kodecellene. Du eksporterer notebooken fra JupyterLab via File-menyen -> Print... I utskriftvinduet velger du «Lagre som PDF» i feltet Destinasjon.

## Problemstilling: Predikere salg av matprodukter (fiktivt eksempel)
Du har fått i oppgave fra Indiske Matgrossisters Forening å utvikle en modell som skal predikere salg av produkter i forskjellige typer matbutikker. De ønsker en modell der de kan oppgi noen karakteristika for produktet og butikken, og så få et estimat for salget. Kunden, det vil si Indiske Matgrossisters Forening, har testdataene. Du får kun utlevert dataene som brukes til trening og validering.

_**Fritekstoppgave**:_

1. _Hva er en god baselinemodell for dette problemet?_

**Svar**:

(Besvar oppgaven her)

## Pakker og nyttige funksjoner

In [ ]:
# Du må først kjøre denne cellen for å importere relevante biblioteker
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Vi velger å ignorere advarsler
import warnings
warnings.filterwarnings('ignore',category = FutureWarning)

from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_validate
import seaborn as sns
from sklearn.preprocessing import StandardScaler,MinMaxScaler
from sklearn.metrics import mean_squared_error,r2_score
import xgboost as xgb

In [ ]:
# Her er hjelpefunksjonene fra modul 1. Du må kjøre denne cellen for å bruke funksjonene.

def make_scatter_plot(data, para1, para2, logx=False, logy=False):
    plt.plot(data[para1], data[para2], '*', alpha=0.1)
    if logx:
        plt.xscale("log")
    if logy:
        plt.yscale('log')
    plt.xlabel(para1)
    plt.ylabel(para2)

def corr_heatmap(df, figsize=(10, 10)):
    _, axs = plt.subplots(figsize=figsize)
    ax = sns.heatmap(
        df, vmin=-1, vmax=1, cmap="BrBG", linewidths=0.5, annot=True, ax=axs)
    ax.set_title('Correlation matrix')
    return ax

def make_histogram(data, variable, binwidth, lower_lim=None, upper_lim=None):
    if upper_lim == None:
        upper_lim = data[variable].max()
    if lower_lim == None:
        lower_lim = data[variable].min()
    bins=np.arange(lower_lim, upper_lim + binwidth, binwidth)
    plt.hist(data[variable],bins=bins,edgecolor = 'black',alpha=0.75,range = [data[variable].min(),upper_lim])
    plt.ylabel("antall")
    plt.xlabel(variable)

## Dataene
Dataene er hentet fra flere ulike matbutikker i India. Vi har disse variablene:
- **ProductID**: En ID som unikt identifiserer produktet. 
- **FatContent**: Om produktet har lavt fettinnhold eller ikke.
- **Visibility**: Hvor synlig produktet er i butikken. Er skalert til å være et tall mellom 0 og 1. 
- **StoreType**: Hva slags type butikk det er. I Norge ville dette ha tilsvart ulike kjeder.
- **Size Store**: Butikkens størrelse gitt som kategori.
- **MRP**: Maximum retail price (https://en.wikipedia.org/wiki/Maximum_retail_price). Merk at MRP kan være forskjellig for samme produkt, avhengig av hvor butikken ligger. 
- **Sales**: Totalt salg av gitt produkt i butikken. Valuta er rupi.

Hver kolonne i datasettet representerer det totale salget av et gitt produkt i en bestemt butikk i 2018.

In [ ]:
# Vi laster dataene inn i en pandas dataframe. Deretter kan vi begynne å jobbe med dataene.

frame=pd.read_csv("data/sales_data.csv")

### Få oversikt over dataene
_**Kodeoppgave 1**:_
- _Få en rask oversikt over dataene._

> **Hint:** Her kan du gjerne bruke pandas' `describe`-funksjon, og du kan bruke `isna`-funksjonen for å finne `NaN`-verdier (dvs. verdier som er «Not a Number»). Koden kan f.eks. se slik ut:
>```
># For numeriske verdier (default setting er å ekskludere kategoriske variabler):
>frame.describe()
>
># For kategoriske verdier:
>frame.describe(exclude=[np.number])
>
># Finn NaN-verdier
>frame.isna().sum()
```

In [ ]:
# Studentkode

# (Skriv din kode her. Lag gjerne flere kodeceller istedenfor å skrive all kode i én celle.)

&nbsp;<br>
_**Fritekstoppgaver**:_

2. _Hvor mange datapunkter har vi?_
3. _Hva er gjennomsnittlig salgspris?_
4. _Hvor mange forskjellige ProductID har vi?_

**Svar**:

(Besvar oppgavene her)

### Utforsk variablene hver for seg
_**Kodeoppgave 2**:_
- _Utforsk alle variablene hver for seg._

>**Hint:**
>Du kan bruke funksjonen `frame.hist()`
>for å raskt lage histogrammer for de numeriske variablene, men bruk gjerne hjelpefunksjonene (som er lagt til toppen av notebooken) for å studere histogrammene i detalj.
>For de kategoriske variablene, er det lurt å bruke `value_counts(dropna=False)` istedenfor vanlig value counts, slik at du også får med verdier som er NaN.

In [ ]:
# Studentkode

# (Skriv din kode her. Lag gjerne flere kodeceller.)

_**Fritekstoppgaver**:_

5. _Du bør nå ha funnet minst to (vi vet om tre) datakvalitetsproblemer. Beskriv dem._
6. _Du kan komme deg rundt et av datakvalitetsproblemene ved å rette opp det som er feil, hvis du har gode grunner til å anta hva det skal rettes til. Hvis du ikke kan gjøre noen slike antagelser, kan du fylle inn gjennomsnittet eller den mest vanlige verdien. Du kan også velge å fjerne alle rader med et gitt datakvalitetproblem. For hver av datakvalitetsproblemene, velg en strategi og begrunn hvorfor._
7. _Du skal ta vekk kolonnen for ProductID. Kan du forklare hvorfor?_

>**Hint 1**: To av problemene har med manglende verdier å gjøre, og ett har med skrivemåter å gjøre.
>
>**Hint 2**:
Nyttige pandas-funksjoner:
>```
># Erstatt et ord med et annet. Kan også brukes for numeriske verdier.
>frame[variabel].replace({"ord1":"nyttord1","ord2":"nyttord2"},inplace=True)
>
># Erstatt NaN-verdier
>frame[variabel].fillna(<tall eller ord som erstatter NaN>,inplace=True)
>
># Ta vekk alle datapunkter som ikke tilfredstiller et krav
>frame=frame[frame[variabel]!=4]
>
># Dropp alle kolonner som inneholder NaN
>frame.dropna(inplace=True)
>
># Dropp en kolonne
>frame.drop(columns=[variabel],inplace=True)
>```

**Svar**:

(Besvar oppgavene her)

### Sjekk korrelasjoner
_**Kodeoppgave 3**:_
- _Sjekk korrelasjoner mellom den avhengige og de uavhengige variabelene._

> **Hint 1**: For å finne korrelasjoner mellom numeriske variabler, er det standard å bruke Pearsons korrelasjonskoeffisient. Denne metoden finner kun linære sammenhenger, men er et godt utgangspunkt. Se notebook i modul 1 for kode.
>
> **Hint 2**: For å finne korrelasjoner mellom kategoriske variabler og den avhengige variabelen, har vi ikke en like god standard. Men en enkel måte å se om en kategorisk variabel påvirker den avhengige variabelen, er å lage et såkalt boksplott for variabelen og se nærmere på de enkelte kategoriene. Generelt viser et slikt boksplott hvordan verdier i et utvalg fordeler seg, med maksverdi, minverdi, medianverdi, 25-persentil og 75-persentil. I vårt tilfelle kan vi tegne opp kategoriene for hver uavhengig variabel, stilt opp mot den avhengige variabelen _Sales_. Hvis vi ser at distribusjonen til _Sales_-dataene for en uavhengig variabel er ganske ulik for de forskjellige kategoriene, er det sannsynlig at denne variabelen påvirker _Sales_.
>
> **Hint 3**: Her er et eksempel på hvordan man lager boksplott for de forskjellige butikkstørrelsene:
>```
>sns.boxplot(x="SizeStore", y="Sales", data=frame)
>```

In [ ]:
# Studentkode

# (Skriv din kode her. Lag gjerne flere kodeceller.)

_**Fritekstoppgave**:_

8. _Hvilke variabler tror du kan komme til å påvirke salget basert på den analysen du har gjort? Begrunn svaret ditt._

**Svar**:

(Besvar oppgaven her)

## Modell

Du skal nå trene en modell som bruker informasjon om et produkt og en butikk til å predikere salget av det produktet i den butikken. Du skal benytte XGBoost og kan starte med å bruke alle de uavhenige variablene (unntatt ProductID, som du allerede har utelukket). Notebook 2 fra modul 2 inneholder både forklaring og eksempel på bruk av XGBoost. Før du trener modellen, må du splitte opp dataene og preprossere dem. Husk at kunden sitter på testdataene, slik at du i dette tilfellet ikke selv trenger å holde av data til testing.

_**Kodeoppgave 4**:_
- _Del inn i trenings- og valideringsdata, og gjør preprosessering av data._

> **Hint**: Tenk på hvilke typer av preprosessering du trenger å gjøre på dataene. Beskrivelse av preprosessering finner du i notebooken til modul 3. I dette tilfellet er det helt greit å gjøre en-hot-koding (one-hot encoding) med den enkle koden vist nedenfor. Det er også greit å gjøre det før du deler opp data, fordi vi antar at alle kategorier vil være representert i både trenings-, validerings- og testdataene.
>```
>frame_processed=pd.get_dummies(frame)
>```

In [ ]:
# Studentkode

# (Skriv din kode her. Lag gjerne flere kodeceller.)

### XGBoost-modell
Når du nå skal lage en XGBoost-modell, kan du starte med å bruke standardverdier for flest mulig av hyperparameterene. Det eneste du skal ta stilling til, er dybden på træerne og evalueringsmetrikken. Her er et kode-eksempel uten hyperparameter-verdier:
```
params = {'max_depth': , 'objective': }
xgb_model = xgb.train(params=params, 
                       dtrain=xgb.DMatrix(X_train, label=y_train))
predicted = xgb_model.predict(xgb.DMatrix(X_eval))
```

_**Kodeoppgave 5**:_
- _Plott prediksjonene og y_eval for å sjekke at modellen predikerer meningsfylt._

> Hint: Du kan f.eks. bruke følgende kode:
> ```
> x = range(len(y_eval))
> plt.scatter(x, y_eval, label='y_eval')
> plt.scatter(x, predicted, label='predicted')
> plt.legend()
> ```

In [ ]:
# Studentkode

# (Skriv din kode her. Lag gjerne flere kodeceller.)

## Modell-evaluering
_**Kodeoppgave 6**:_
- _Evaluer modellen din med RMSE og R2._

> Hint: Du kan benytte funksjonene `mean_squared_error` og `r2_score` fra scikit-learn som vi lastet inn på starten. Se eksempler i modul 3.

In [ ]:
# Studentkode

# (Skriv din kode her. Lag gjerne flere kodeceller.)

_**Fritekstoppgaver**:_

9. _Beskriv minst én fordel og én ulempe ved XGBoost._
10. _Forklar forskjellen på RMSE og R2._
11. _Fungerte modellen like godt på valideringsdataene som på treningsdataene? Hvis ikke, forklar hvorfor. Kan du bruke dette til å forbedre modellen?_ 

**Svar**:

(Besvar oppgavene her)

### Eksperimenter med å ta vekk variabler
_**Kodeoppgave 7**:_
- _Forsøk å ta vekk variabler du tror ikke er signifikante basert på den utforskende analysen._

> Hint: Du finner eksempler i modul 3.

In [ ]:
# Studentkode

# (Skriv din kode her. Lag gjerne flere kodeceller.)

_**Fritekstoppgaver**:_

12. _Bør du ta vekk variablene som du tenkte ikke var signifikante?_
13. _Hva kan gjøres for å forbedre modellen?_

**Svar**:

(Besvar oppgavene her)